# OCR BNP — Pipeline V13 — Qwen3.6-27B-FP8 / H100 / Offline Domino
> **Ordre recommandé :** `Kernel → Restart Kernel and Run All Cells`

**V13 :**
- Qwen3.6-27B-FP8 à la place de Qwen2.5-VL-7B-Instruct
- kernel `finegrained-fp8` v4 chargé localement (Domino offline)
- `AutoModelForMultimodalLM`
- BF16 activations / poids FP8 selon le checkpoint
- checkpoint JSON : un PDF déjà traité n'est pas retraité
- suivi tokens + temps + progression
- correction des erreurs de syntaxe présentes dans V12


## 1. Packages nécessaires — À conserver pour les prochains environnements

### Stack validée sur Domino
- Python 3.11.x
- NVIDIA H100 80GB
- `torch==2.11.0` + CUDA wheel `cu130`
- `torchvision==0.26.0`
- `torchaudio==2.11.0`
- `transformers==5.14.1`
- `kernels==0.15.2`
- `triton==3.6.0` (installé avec la stack Torch)
- `accelerate`
- `safetensors`
- `huggingface-hub`
- `PyMuPDF`
- `Pillow`
- `numpy`
- `pandas`
- `openpyxl`
- `psutil`

### Commandes terminal — nouvel environnement
```bash
python -m pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu130
python -m pip install transformers==5.14.1 kernels==0.15.2 accelerate safetensors huggingface-hub PyMuPDF Pillow numpy pandas openpyxl psutil
```

### Dépendance FP8 locale obligatoire
Le dossier suivant doit être présent :
```text
/mnt/finegrained-fp8/build/torch-cuda
```
Il doit correspondre à `kernels-community/finegrained-fp8` **v4** et exposer :
`matmul_2d`, `matmul_batched`, `matmul_grouped`.


In [ ]:
# Vérification des packages — cette cellule n'installe rien
import sys, importlib

required = {
    "torch": "2.11.0",
    "transformers": "5.14.1",
    "kernels": "0.15.2",
    "fitz": "PyMuPDF",
    "PIL": "Pillow",
    "numpy": "numpy",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "psutil": "psutil",
    "accelerate": "accelerate",
    "safetensors": "safetensors",
}

for module, label in required.items():
    try:
        m = importlib.import_module(module)
        version = getattr(m, "__version__", "OK")
        print(f"✅ {label:18s} {version}")
    except Exception as e:
        print(f"❌ {label:18s} MANQUANT — {e}")

print("Python             :", sys.version.split()[0])


## 2. Imports

In [ ]:
import os, sys, time, json, re, gc, importlib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz
import torch
import psutil
from PIL import Image
from transformers import AutoProcessor, AutoModelForMultimodalLM
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('✅ Imports OK')
print('Torch        :', torch.__version__)
print('CUDA Torch   :', torch.version.cuda)
print('Transformers :', __import__('transformers').__version__)
print('GPU          :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun')


## 3. Config

In [ ]:
PIPELINE_VERSION = "OCR_V13_QWEN36_27B_FP8"

MODEL_PATH = Path("/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main")
LOCAL_FP8_KERNEL_PATH = Path("/mnt/finegrained-fp8/build/torch-cuda")

DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS  = 700
IMAGE_MAX_SIZE  = 1120
PDF_ZOOM        = 2.0
BLANK_THRESHOLD = 0.97
GPU_BATCH_SIZE  = 1   # Démarrage prudent pour le 27B FP8. Monter à 2 après validation VRAM/vitesse.

INPUT_DIR  = Path("/mnt/data/transferts_in")
OUTPUT_DIR = Path("/mnt/data/transferts_out")
JSON_DIR   = OUTPUT_DIR / "json_dossiers"
LOG_PATH   = OUTPUT_DIR / "pipeline.log"
EXCEL_PATH = OUTPUT_DIR / f"audit_transferts_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

assert MODEL_PATH.exists(), f"Modèle introuvable : {MODEL_PATH}"
assert LOCAL_FP8_KERNEL_PATH.exists(), f"Kernel FP8 local introuvable : {LOCAL_FP8_KERNEL_PATH}"
assert torch.cuda.is_available(), "GPU CUDA non disponible"

pdfs = sorted(INPUT_DIR.glob("*.pdf"))
print(f'Pipeline        : {PIPELINE_VERSION}')
print(f'Device          : {DEVICE}')
print(f'GPU             : {torch.cuda.get_device_name(0)}')
print(f'GPU batch size  : {GPU_BATCH_SIZE}')
print(f'Dossiers        : {len(pdfs)}')
print(f'Modèle          : {MODEL_PATH}')
print(f'Kernel FP8      : {LOCAL_FP8_KERNEL_PATH}')
print(f'JSON dir        : {JSON_DIR}')
print(f'Excel           : {EXCEL_PATH}')


## 4. Kernel FP8 local + chargement Qwen3.6-27B-FP8


In [ ]:
# IMPORTANT : charger le kernel FP8 local AVANT le modèle Qwen3.6
os.environ["USE_HUB_KERNELS"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR"] = "1"

kernel_path_str = str(LOCAL_FP8_KERNEL_PATH)
if kernel_path_str not in sys.path:
    sys.path.insert(0, kernel_path_str)

local_fp8_kernel = importlib.import_module("finegrained_fp8")
for fn in ("matmul_2d", "matmul_batched", "matmul_grouped"):
    assert hasattr(local_fp8_kernel, fn), f"Fonction FP8 manquante : {fn}"

# Force Transformers à utiliser le kernel local au lieu du Hub
import transformers.integrations.finegrained_fp8 as tf_fp8
tf_fp8.lazy_load_kernel = lambda *args, **kwargs: local_fp8_kernel
tf_fp8._load_finegrained_fp8_kernel.cache_clear()
_fp8_check = tf_fp8._load_finegrained_fp8_kernel()
print('✅ Kernel finegrained-fp8 local raccordé à Transformers')

# Processor + modèle local
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
)

print('Chargement Qwen3.6-27B-FP8...')
model = AutoModelForMultimodalLM.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
    device_map="auto",
    dtype="auto",
)
model.eval()

print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')
print('Type          :', type(model).__name__)
print('Device        :', next(model.parameters()).device)
print('Dtype         :', next(model.parameters()).dtype)
print('VRAM allouée  :', round(torch.cuda.memory_allocated()/1024**3, 2), 'GB')


## 5. Utilitaires PDF & inférence Qwen3.6


In [ ]:
def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)


def is_blank(image, threshold=BLANK_THRESHOLD) -> bool:
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold


def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    pages = []
    with fitz.open(path) as doc:
        matrix = fitz.Matrix(zoom, zoom)
        for i in range(len(doc)):
            pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
            img = resize(Image.frombytes('RGB', [pix.width, pix.height], pix.samples))
            pages.append({'index': i, 'image': img})
    return pages


def parse_json(text: str) -> dict:
    try:
        m = re.search(r'\{.*\}', text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}


def _build_inputs(prompt: str, image: Image.Image):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': prompt},
        ]
    }]
    # Qwen3.6 : thinking désactivé pour extraction structurée
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors='pt',
            enable_thinking=False,
        ).to(model.device)
    except TypeError:
        # Compatibilité si le processor local ne reconnaît pas enable_thinking
        return processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors='pt',
        ).to(model.device)


def ask_single(prompt: str, image: Image.Image) -> dict:
    inputs = _build_inputs(prompt, image)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )
    elapsed = time.time() - t0
    in_len = inputs['input_ids'].shape[1]
    generated = out[0][in_len:]
    text = processor.batch_decode(
        generated.unsqueeze(0),
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    attn = inputs.get('attention_mask')
    tok_in = int(attn[0].sum().item()) if attn is not None else int(in_len)
    return {
        'text': text,
        'tokens_in': tok_in,
        'tokens_out': int(len(generated)),
        'elapsed': round(elapsed, 2),
    }


def ask_batch(prompt: str, images: list) -> list:
    # V13 : mode robuste. Le 27B traite chaque page séquentiellement.
    # GPU_BATCH_SIZE contrôle le groupement de progression, pas un vrai batch tensoriel.
    return [ask_single(prompt, img) for img in images]


print('✅ Utilitaires Qwen3.6 OK')


## 6. Normalisation des données

In [ ]:
def norm_str(v):
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null', 'none', 'n/a') else None

def norm_upper(v):
    s = norm_str(v)
    return s.upper() if s else None

def norm_compte(v):
    s = norm_str(v)
    return re.sub(r'[^A-Za-z0-9]', '', s).upper() if s else None

def norm_montant(v):
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = re.sub(r'[^\d.,-]', '', str(v).strip())
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif '.' in s and ',' in s: s = s.replace(',', '')
    elif s.count(',') > 1: s = s.replace(',', '')
    try: return float(s)
    except Exception: return None

def norm_date(v):
    s = norm_str(v)
    if not s: return None
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s): return s
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', s)
    return f'{m.group(3)}/{m.group(2)}/{m.group(1)}' if m else s

def norm_periode(v):
    s = norm_str(v)
    if not s: return None
    s = s.upper()
    s = re.sub(r'[._\-]', ' ', s)
    s = re.sub(r'PART\s*(\d)', r'PART \1', s)
    return re.sub(r'\s+', ' ', s).strip()

CHAMPS_OV = {
    'type','monnaie','montant_chiffres','montant_lettres','periode','mois','annee',
    'tranche','complement_ov','date_demande','compte_donneur_ordre',
    'nature_paiement_case','nature_paiement_autre_libelle','beneficiaire_nom',
    'beneficiaire_compte','beneficiaire_adresse','code_swift_banque_beneficiaire',
    'nom_banque_beneficiaire'
}
CHAMPS_ANN1 = {
    'type','nom_prenom_employe','date_naissance','résidence','compte_bancaire_local',
    'nom_prenom_signataire','date_signature'
}
CHAMPS_ANN2 = {
    'type','mois_transfert','Periode_transfert','tranche_transfert','complement_transfert',
    'nom_prenom_travailleur','compte_bancaire_local','salaire_mensuel','nombre_jours',
    'nombre_jours_absence','part_transferable','pays_destination','nom_banque_etranger',
    'numero_compte_devise_etranger','numero_domiciliation'
}
CHAMPS_BP = {
    'type','nom_prenom_salarie','matricule','mois_bulletin','salaire_base','salaire_brut',
    'retenue_ss','retenue_irg','retenue_mutuelle','net_a_payer'
}

def normalise_doc(doc_type: str, data: dict) -> dict:
    d = dict(data or {})
    if doc_type == 'OV':
        for key in list(d.keys()):
            if isinstance(d.get(key), dict): d.update(d.pop(key))
        if not d.get('compte_donneur_ordre'):
            for alias in ['compte_donneur','compte_debiteur','compte_bancaire','Siége Racine Ordinal clé','N°DOM','numero_compte','compte_debiteur_ordre']:
                if d.get(alias):
                    d['compte_donneur_ordre'] = d[alias]; break
        if d.get('tranche') in ('70','32','50','59','57',70,32,50,59,57): d['tranche'] = None
        if d.get('periode') and re.match(r'^\d{2}/\d{2}/\d{4}$', str(d['periode'])): d['periode'] = None
        d = {k:v for k,v in d.items() if k in CHAMPS_OV}
        d['monnaie'] = norm_upper(d.get('monnaie'))
        d['montant_chiffres'] = norm_montant(d.get('montant_chiffres'))
        d['montant_lettres'] = norm_str(d.get('montant_lettres'))
        d['periode'] = norm_periode(d.get('periode'))
        d['mois'] = norm_periode(d.get('mois'))
        d['annee'] = norm_str(d.get('annee'))
        d['tranche'] = norm_str(d.get('tranche'))
        d['complement_ov'] = norm_str(d.get('complement_ov'))
        d['date_demande'] = norm_date(d.get('date_demande'))
        d['compte_donneur_ordre'] = norm_compte(d.get('compte_donneur_ordre'))
        d['nature_paiement_case'] = norm_str(d.get('nature_paiement_case'))
        d['nature_paiement_autre_libelle'] = norm_str(d.get('nature_paiement_autre_libelle'))
        d['beneficiaire_nom'] = norm_upper(d.get('beneficiaire_nom'))
        d['beneficiaire_compte'] = norm_compte(d.get('beneficiaire_compte'))
        d['beneficiaire_adresse'] = norm_str(d.get('beneficiaire_adresse'))
        d['code_swift_banque_beneficiaire'] = norm_compte(d.get('code_swift_banque_beneficiaire'))
        d['nom_banque_beneficiaire'] = norm_upper(d.get('nom_banque_beneficiaire'))
    elif doc_type == 'ANNEXE_I':
        d = {k:v for k,v in d.items() if k in CHAMPS_ANN1}
        d['nom_prenom_employe'] = norm_upper(d.get('nom_prenom_employe'))
        d['date_naissance'] = norm_date(d.get('date_naissance'))
        d['résidence'] = norm_str(d.get('résidence'))
        d['compte_bancaire_local'] = norm_compte(d.get('compte_bancaire_local'))
        d['nom_prenom_signataire'] = norm_upper(d.get('nom_prenom_signataire'))
        d['date_signature'] = norm_date(d.get('date_signature'))
    elif doc_type == 'ANNEXE_II':
        d = {k:v for k,v in d.items() if k in CHAMPS_ANN2}
        d['Periode_transfert'] = norm_periode(d.get('Periode_transfert') or d.get('mois_transfert'))
        d['tranche_transfert'] = norm_str(d.get('tranche_transfert'))
        d['complement_transfert'] = norm_str(d.get('complement_transfert'))
        d['nom_prenom_travailleur'] = norm_upper(d.get('nom_prenom_travailleur'))
        d['compte_bancaire_local'] = norm_compte(d.get('compte_bancaire_local'))
        d['salaire_mensuel'] = norm_montant(d.get('salaire_mensuel'))
        d['nombre_jours'] = norm_str(d.get('nombre_jours'))
        d['nombre_jours_absence'] = norm_str(d.get('nombre_jours_absence'))
        d['part_transferable'] = norm_montant(d.get('part_transferable'))
        d['pays_destination'] = norm_upper(d.get('pays_destination'))
        d['nom_banque_etranger'] = norm_str(d.get('nom_banque_etranger'))
        d['numero_compte_devise_etranger'] = norm_compte(d.get('numero_compte_devise_etranger'))
        d['numero_domiciliation'] = norm_str(d.get('numero_domiciliation'))
    elif doc_type == 'BULLETIN':
        d = {k:v for k,v in d.items() if k in CHAMPS_BP}
        d['nom_prenom_salarie'] = norm_upper(d.get('nom_prenom_salarie'))
        d['matricule'] = norm_str(d.get('matricule'))
        d['mois_bulletin'] = norm_periode(d.get('mois_bulletin'))
        for k in ['salaire_base','salaire_brut','retenue_ss','retenue_irg','retenue_mutuelle','net_a_payer']:
            d[k] = norm_montant(d.get(k))
    return d

print('✅ Normalisation OK')


## 7. Prompt universel

In [ ]:
PROMPT_UNIVERSEL = r"""
Lis ce document bancaire.

ÉTAPE 1 — Identifie le type en lisant le titre principal :
- OV : titre « ORDRE DE VIREMENT A L'ETRANGER ».
- ANNEXE_II : titre « Annexe II », contient « Fiche de paie spéciale relative à un transfert du salaire ».
- BULLETIN : titre « BULLETIN DE PAIE ».
- ANNEXE_I : titre exactement « Annexe I » (JAMAIS « ANNEXE II »). Lettre adressée au Directeur de l'agence BNP, commençant par « Je soussigné... », sans tableau de salaire.
- AUTRE : tout autre document (page vide, tableau sans rapport, email, etc.).

ÉTAPE 2 — Selon le type identifié, extrais uniquement les champs listés ci-dessous.
Si AUTRE : retourne uniquement {"type":"AUTRE"}.

--- Si OV ---
monnaie, montant_chiffres, montant_lettres,
periode (mois + année, jamais JJ/MM/AAAA ; ex. JANVIER 2026 ; null si absent),
mois, annee, tranche (P1/P2/PART 1/PART 2 ; null si absent), complement_ov (COM/COMP/COMPLEMENT ; null si absent),
date_demande, compte_donneur_ordre (zone « Siege Racine Ordinal clé » ou « N°DOM »),
nature_paiement_case (une seule valeur parmi virement_commercial, virement_trésorerie, urgent, non urgent),
nature_paiement_autre_libelle (texte exact entre parenthèses à côté de « Autre », sinon null),
beneficiaire_nom, beneficiaire_compte (IBAN sans espaces), beneficiaire_adresse,
code_swift_banque_beneficiaire, nom_banque_beneficiaire.

--- Si ANNEXE_II ---
Periode_transfert (valeur brute après « Mois de »),
tranche_transfert (P1/P2/PART 1/PART 2 si visible, sinon null),
complement_transfert (COM/COMP/COMPLEMENT si visible, sinon null),
nom_prenom_travailleur, compte_bancaire_local (20 chiffres), salaire_mensuel,
nombre_jours (après le libellé exact « Nombre de jour »),
nombre_jours_absence (après le libellé exact « Nombre de jour d'absence »),
part_transferable, pays_destination, nom_banque_etranger,
numero_compte_devise_etranger,
numero_domiciliation : lire la ligne du tableau BNP sous « DOMICILIATION IMPORT ». Retourner les 5 valeurs séparées par | exactement comme visibles. Exemple : 271901|YYYY.N|NN|NNNNN|DZD. Null si absent/vide/illisible.

--- Si ANNEXE_I ---
nom_prenom_employe (juste après « Je soussigné »),
date_naissance (après « Né le »), résidence (après le libellé de résidence),
compte_bancaire_local (20 chiffres), nom_prenom_signataire,
date_signature (après « Bethioua le », format jj/MM/aaaa).

--- Si BULLETIN ---
nom_prenom_salarie, matricule, mois_bulletin,
salaire_base, salaire_brut, retenue_ss, retenue_irg, retenue_mutuelle, net_a_payer.

RÈGLES :
- Retourne UNIQUEMENT un JSON valide, sans texte avant ni après.
- Commence toujours par {"type":"..."}.
- Si un champ est absent ou illisible : null.
- Pas de markdown, pas de backticks.
- N'invente aucun champ supplémentaire.
- Recopie les valeurs brutes visibles ; ne reformule pas et ne traduis pas.
- Pour un texte entre parenthèses demandé, conserve exactement le contenu visible.
"""

print(f'✅ Prompt : {len(PROMPT_UNIVERSEL)} caractères')


## 8. Traitement d'un PDF (batch GPU)

In [ ]:
def process_pdf(pdf_path: Path, verbose=True) -> dict:
    """
    Traite un PDF avec batch GPU.
    Toutes les pages non-blanches sont envoyées par groupes de GPU_BATCH_SIZE.
    Retourne un dict avec données + stats (tokens, temps).
    """
    import gc
    TYPES_ATTENDUS = {'OV', 'ANNEXE_II', 'BULLETIN', 'ANNEXE_I'}

    pages    = pdf_to_pages(pdf_path)
    results  = {}
    doublons = []
    total_tok_in  = 0
    total_tok_out = 0
    t_dossier     = time.time()

    if verbose:
        print(f'\n📁 {pdf_path.name} — {len(pages)} page(s)')

    # Filtrer pages non-blanches
    pages_actives = [p for p in pages if not is_blank(p['image'])]
    pages_vides   = len(pages) - len(pages_actives)

    if verbose and pages_vides:
        print(f'  {pages_vides} page(s) vide(s) ignorée(s)')

    # Traitement par batch
    for batch_start in range(0, len(pages_actives), GPU_BATCH_SIZE):
        batch  = pages_actives[batch_start:batch_start + GPU_BATCH_SIZE]
        images = [p['image'] for p in batch]

        t_batch = time.time()
        reps    = ask_batch(PROMPT_UNIVERSEL, images)
        elapsed_batch = time.time() - t_batch

        for page, rep in zip(batch, reps):
            i        = page['index']
            data     = parse_json(rep['text'])
            doc_type = data.get('type', 'INCONNU')

            total_tok_in  += rep['tokens_in']
            total_tok_out += rep['tokens_out']

            if doc_type in ('AUTRE', 'INCONNU'):
                if verbose:
                    print(f'  Page {i+1} → {doc_type} — ignorée')
                continue

            data = normalise_doc(doc_type, data)

            if verbose:
                print(f'  Page {i+1} → {doc_type:10s} '
                      f'| tok={rep["tokens_in"]}+{rep["tokens_out"]}')
                for k, v in data.items():
                    if k != 'type' and v is not None:
                        print(f'    {k:25s}: {v}')

            if doc_type in results:
                doublons.append(doc_type)
                if verbose: print(f'    ⚠️  DOUBLON {doc_type}')
                continue

            results[doc_type] = data

        # Purge VRAM entre batches
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pages_trouvees   = sorted(results.keys())
    pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

    return {
        'pipeline_version': PIPELINE_VERSION,
        'fichier':          pdf_path.name,
        'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'temps_total_s':    round(time.time() - t_dossier, 2),
        'tokens_in':        total_tok_in,
        'tokens_out':       total_tok_out,
        'tokens_total':     total_tok_in + total_tok_out,
        'pages_trouvees':   ', '.join(pages_trouvees),
        'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
        'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
        **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
    }


print('✅ process_pdf OK')


## 9. Export Excel

In [ ]:
SCHEMA = {
    'META': [
        ('fichier','Fichier'), ('date_traitement','Date traitement'), ('temps_total_s','Temps (s)'),
        ('tokens_total','Tokens total'), ('pages_trouvees','Pages trouvées'),
        ('pages_manquantes','Pages manquantes'), ('anomalies','Anomalies'),
    ],
    'OV': [
        ('monnaie','Monnaie'), ('montant_chiffres','Montant (chiffres)'), ('montant_lettres','Montant (lettres)'),
        ('periode','Période'), ('mois','Mois'), ('annee','Année'), ('tranche','Tranche'), ('complement_ov','Complément'),
        ('date_demande','Date demande'), ('compte_donneur_ordre','Compte donneur ordre'),
        ('nature_paiement_case','Nature paiement'), ('nature_paiement_autre_libelle','Devise / autre libellé'),
        ('beneficiaire_nom','Bénéficiaire nom'), ('beneficiaire_compte','Bénéficiaire compte'),
        ('beneficiaire_adresse','Bénéficiaire adresse'), ('code_swift_banque_beneficiaire','SWIFT'),
        ('nom_banque_beneficiaire','Banque bénéficiaire'),
    ],
    'ANNEXE_I': [
        ('nom_prenom_employe','Nom Prénom client'), ('date_naissance','Date naissance'), ('résidence','Résidence'),
        ('compte_bancaire_local','Compte bancaire'), ('nom_prenom_signataire','Nom Prénom signataire'),
        ('date_signature','Date signature'),
    ],
    'ANNEXE_II': [
        ('Periode_transfert','Mois transfert'), ('tranche_transfert','Tranche transfert'),
        ('complement_transfert','Complément transfert'), ('nom_prenom_travailleur','Nom Prénom'),
        ('compte_bancaire_local','Compte bancaire'), ('salaire_mensuel','Salaire mensuel'),
        ('nombre_jours','Nombre de jours'), ('nombre_jours_absence',"Nombre jours d'absence"),
        ('part_transferable','Part transférable'), ('pays_destination','Pays destination'),
        ('nom_banque_etranger','Nom banque'), ('numero_compte_devise_etranger','Compte étranger'),
        ('numero_domiciliation','N° Domiciliation'),
    ],
    'BULLETIN': [
        ('nom_prenom_salarie','Nom Prénom'), ('matricule','Matricule'), ('mois_bulletin','Mois bulletin'),
        ('salaire_base','Salaire base'), ('salaire_brut','Salaire brut'), ('retenue_ss','Retenue SS'),
        ('retenue_irg','Retenue IRG'), ('retenue_mutuelle','Retenue mutuelle'), ('net_a_payer','Net à payer'),
    ],
}

COLORS = {
    'META': {'header':'FF1F4E79','col':'FFD6E4F0'}, 'OV': {'header':'FF833C00','col':'FFFCE4D6'},
    'ANNEXE_I': {'header':'FF375623','col':'FFE2EFDA'}, 'ANNEXE_II': {'header':'FF203864','col':'FFDAE3F3'},
    'BULLETIN': {'header':'FF3F3151','col':'FFEDE7F6'},
}

def create_excel(path: Path, rows: list):
    wb = Workbook(); ws = wb.active; ws.title = 'Dossiers'
    all_cols = [(g,k,l) for g,cols in SCHEMA.items() for k,l in cols]
    group_map = defaultdict(list)
    for idx,(g,_,_) in enumerate(all_cols,1): group_map[g].append(idx)
    for g,cols in group_map.items():
        s,e=cols[0],cols[-1]
        if s<e: ws.merge_cells(start_row=1,start_column=s,end_row=1,end_column=e)
        c=ws.cell(1,s,g); c.font=Font(bold=True,color='FFFFFFFF',name='Arial',size=11)
        c.fill=PatternFill('solid',start_color=COLORS[g]['header']); c.alignment=Alignment(horizontal='center')
    for i,(g,_,label) in enumerate(all_cols,1):
        c=ws.cell(2,i,label); c.font=Font(bold=True,name='Arial',size=9)
        c.fill=PatternFill('solid',start_color=COLORS[g]['col']); c.alignment=Alignment(horizontal='center',wrap_text=True)
        c.border=Border(bottom=Side(style='thin'),right=Side(style='hair')); ws.column_dimensions[get_column_letter(i)].width=20
    ws.freeze_panes=ws.cell(3,len(SCHEMA['META'])+1)
    for r,dossier in enumerate(rows,3):
        for cidx,(g,key,_) in enumerate(all_cols,1):
            val=dossier.get(key) if g=='META' else (dossier.get(g) or {}).get(key)
            c=ws.cell(r,cidx,val); c.font=Font(name='Arial',size=9); c.fill=PatternFill('solid',start_color=COLORS[g]['col'])
            if isinstance(val,(int,float)) and key not in ('tokens_total',): c.number_format='0.00'
            if g=='META' and key in ('anomalies','pages_manquantes') and val: c.font=Font(name='Arial',size=9,bold=True,color='FFCC0000')
    wb.save(path)
    print(f'✅ Excel : {path} | {len(rows)} dossiers | {len(all_cols)} colonnes')

print('✅ Export Excel OK')


## 10. Log

In [ ]:
def log(msg: str):
    ligne = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + '\n')


print('✅ Log OK')

## 11. Pipeline complet

In [ ]:
import gc
import psutil

# ── Vérification RAM ──────────────────────────────────────────────────────────
ram_libre = psutil.virtual_memory().available / 1e9
print(f'RAM libre : {ram_libre:.1f} GB')

# ── Dossiers à traiter ────────────────────────────────────────────────────────
deja_traites = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter    = [p for p in pdfs if p.stem not in deja_traites]
total_pdfs   = len(pdfs)

log(f'Total PDFs     : {total_pdfs}')
log(f'Déjà traités   : {len(deja_traites)}')
log(f'À traiter      : {len(a_traiter)}')
log(f'GPU batch size : {GPU_BATCH_SIZE}')
log(f'RAM libre      : {ram_libre:.1f} GB')

# ── Chunk size automatique selon RAM disponible ───────────────────────────────
taille_img_mb = 1120 * 1584 * 3 / 1e6       # ~5MB par image
ram_estimee   = len(a_traiter) * 4 * taille_img_mb / 1e3
CHUNK_SIZE    = min(
    len(a_traiter),
    max(GPU_BATCH_SIZE, int(ram_libre * 0.7 * 1e3 / (4 * taille_img_mb)))
)
log(f'RAM estimée    : {ram_estimee:.1f} GB')
log(f'Chunk size     : {CHUNK_SIZE} dossiers')

chunks = [a_traiter[i:i+CHUNK_SIZE] for i in range(0, len(a_traiter), CHUNK_SIZE)]
log(f'{len(chunks)} chunk(s) de {CHUNK_SIZE} dossiers max')

# ── Pipeline ──────────────────────────────────────────────────────────────────
t_total       = time.time()
grand_tok_in  = 0
grand_tok_out = 0
n_ok = n_err  = 0
TYPES_ATTENDUS = {'OV', 'ANNEXE_I', 'ANNEXE_II', 'BULLETIN'}

for num_chunk, chunk in enumerate(chunks, start=1):
    log(f'── Chunk {num_chunk}/{len(chunks)} : {len(chunk)} dossiers ──')

    # ── Phase 1 : Chargement pages ────────────────────────────────────────────
    t_load    = time.time()
    all_pages = []
    for pdf_path in chunk:
        try:
            pages = pdf_to_pages(pdf_path)
            for page in pages:
                if not is_blank(page['image']):
                    all_pages.append((pdf_path, page))
        except Exception as e:
            log(f'❌ Chargement {pdf_path.name} : {e}')
    log(f'   {len(all_pages)} pages chargées en {time.time()-t_load:.1f}s')

    # ── Phase 2 : Inférence GPU ───────────────────────────────────────────────
    t_infer     = time.time()
    raw_results = defaultdict(list)

    for batch_start in range(0, len(all_pages), GPU_BATCH_SIZE):
        batch  = all_pages[batch_start:batch_start + GPU_BATCH_SIZE]
        images = [page['image'] for _, page in batch]
        try:
            reps = ask_batch(PROMPT_UNIVERSEL, images)
            for (pdf_path, page), rep in zip(batch, reps):
                raw_results[pdf_path].append({
                    'index':      page['index'],
                    'text':       rep['text'],
                    'tokens_in':  rep['tokens_in'],
                    'tokens_out': rep['tokens_out'],
                })
            pct = min(100, (batch_start + len(batch)) / len(all_pages) * 100)
            log(f'   {batch_start+len(batch):>5}/{len(all_pages)} pages | {pct:.0f}%')
        except Exception as e:
            log(f'❌ Batch {batch_start}-{batch_start+len(batch)} : {e}')
            continue
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    log(f'   Inférence en {time.time()-t_infer:.1f}s')

    # ── Phase 3 : Post-traitement + sauvegarde JSON ───────────────────────────
    t_post = time.time()

    for num, pdf_path in enumerate(chunk, start=1):
        try:
            pages_raw = raw_results.get(pdf_path, [])
            if not pages_raw:
                log(f'   [{num:>4}/{len(chunk)}] ⚠️  {pdf_path.name} — aucune page')
                continue

            results  = {}
            doublons = []
            total_tok_in  = 0
            total_tok_out = 0
            t_dossier = time.time()

            for page_raw in sorted(pages_raw, key=lambda x: x['index']):
                data     = parse_json(page_raw['text'])
                doc_type = data.get('type', 'INCONNU')
                total_tok_in  += page_raw['tokens_in']
                total_tok_out += page_raw['tokens_out']
                if doc_type in ('AUTRE', 'INCONNU'):
                    continue
                data = normalise_doc(doc_type, data)
                if doc_type in results:
                    doublons.append(doc_type)
                    continue
                results[doc_type] = data

            pages_trouvees   = sorted(results.keys())
            pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

            result = {
                'pipeline_version': PIPELINE_VERSION,
                'fichier':          pdf_path.name,
                'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'temps_total_s':    round(time.time() - t_dossier, 2),
                'tokens_in':        total_tok_in,
                'tokens_out':       total_tok_out,
                'tokens_total':     total_tok_in + total_tok_out,
                'pages_trouvees':   ', '.join(pages_trouvees),
                'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
                'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
                **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
            }

            json_file = JSON_DIR / f'{pdf_path.stem}.json'
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(result, f, ensure_ascii=False, indent=2, default=str)

            grand_tok_in  += total_tok_in
            grand_tok_out += total_tok_out
            n_ok += 1

            msg = (f'   [{num:>4}/{len(chunk)}] ✅ {pdf_path.name}'
                   f' | tok={result["tokens_total"]}'
                   f' | {result["pages_trouvees"]}')
            if result.get('pages_manquantes'):
                msg += f' | ⚠️  {result["pages_manquantes"]}'
            if result.get('anomalies'):
                msg += f' | 🔴 {result["anomalies"]}'
            log(msg)

        except Exception as e:
            n_err += 1
            log(f'   [{num:>4}/{len(chunk)}] ❌ {pdf_path.name} — {e}')
            continue

    log(f'   Post-traitement en {time.time()-t_post:.1f}s')

    # Libérer RAM du chunk
    del all_pages, raw_results
    gc.collect()
    ram_now = psutil.virtual_memory().available / 1e9
    log(f'   RAM libre après chunk : {ram_now:.1f} GB')

# ── Reconstruction Excel ──────────────────────────────────────────────────────
log('Génération Excel...')
rows = []
for json_file in sorted(JSON_DIR.glob('*.json')):
    with open(json_file, encoding='utf-8') as f:
        rows.append(json.load(f))

create_excel(EXCEL_PATH, rows)

elapsed = time.time() - t_total
log(f'✅ Terminé en {elapsed:.1f}s ({elapsed/max(1,n_ok):.1f}s/dossier)')
log(f'   Traités      : {n_ok} | Erreurs : {n_err}')
log(f'   Tokens IN    : {grand_tok_in:,}')
log(f'   Tokens OUT   : {grand_tok_out:,}')
log(f'   Tokens TOTAL : {grand_tok_in + grand_tok_out:,}')


## 12. Récupération d'urgence

Si le kernel plante et que `rows` est encore en mémoire, exécuter cette cellule
pour sauvegarder immédiatement en JSON avant de tout perdre.

In [ ]:
# Décommenter et exécuter en cas de crash kernel
# try:
#     print(f'rows en mémoire : {len(rows)}')
#     for result in rows:
#         nom = Path(result['fichier']).stem
#         jf  = JSON_DIR / f'{nom}.json'
#         if not jf.exists():
#             with open(jf, 'w', encoding='utf-8') as f:
#                 json.dump(result, f, ensure_ascii=False, indent=2, default=str)
#     print(f'✅ Sauvegardé')
# except Exception as e:
#     print(f'rows non disponible : {e}')

## 13. Analyse des résultats

In [ ]:
import pandas as pd

if EXCEL_PATH.exists():
    df = pd.read_excel(EXCEL_PATH, header=1)
    print(f'📊 {len(df)} dossiers')

    # Stats tokens
    if 'Tokens total' in df.columns:
        print(f'   Tokens total   : {df["Tokens total"].sum():,.0f}')
        print(f'   Tokens/dossier : {df["Tokens total"].mean():,.0f}')
    if 'Temps (s)' in df.columns:
        print(f'   Temps moyen    : {df["Temps (s)"].mean():.1f}s/dossier')
    print()

    if 'Anomalies' in df.columns:
        anom = df[df['Anomalies'].notna()]
        print(f'🔴 Doublons : {len(anom)}')
        if len(anom): print(anom[['Fichier','Anomalies']].to_string(index=False))
        print()

    if 'Pages manquantes' in df.columns:
        manq = df[df['Pages manquantes'].notna()]
        print(f'⚠️  Incomplets : {len(manq)}')
        if len(manq): print(manq[['Fichier','Pages manquantes']].to_string(index=False))
        print()

    display(df.head(5))
else:
    print('Excel non encore généré.')